# 정상 배치 vs Fault 배치 비교

시간별 행이 아닌 100개 배치를 분석 단위로 사용한다. 정상 90개와 Fault 10개의 배치 성과·공정 요약값을 Welch t-test와 Mann–Whitney U로 비교하고, 다중검정에는 FDR 보정을 적용한다. 원본 데이터는 변경하지 않으며 CSV를 저장하지 않는다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


def fdr_bh(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    result = np.empty_like(adjusted)
    result[order] = np.clip(adjusted, 0, 1)
    return result


def hedges_g(sample_a, sample_b):
    sample_a, sample_b = np.asarray(sample_a), np.asarray(sample_b)
    n_a, n_b = len(sample_a), len(sample_b)
    pooled_var = ((n_a - 1) * sample_a.var(ddof=1) + (n_b - 1) * sample_b.var(ddof=1)) / (n_a + n_b - 2)
    correction = 1 - 3 / (4 * (n_a + n_b) - 9)
    return correction * (sample_a.mean() - sample_b.mean()) / np.sqrt(pooled_var)


data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
print(f'{data.shape[0]:,}행 × {data.shape[1]}열, {data["배치번호"].nunique()}개 배치')

113,935행 × 39열, 100개 배치


### 확인

입력은 113,935행 × 39열이며 배치는 100개다. 통계검정은 이 100개 배치를 독립 표본으로 사용한다.

In [2]:
def slope(x, y):
    return float(np.polyfit(x, y, 1)[0])


batch_rows = []
for batch_number, batch in data.groupby('배치번호', sort=True):
    batch = batch.sort_values('발효시간(h)')
    time = batch['발효시간(h)'].to_numpy()
    end_time = time[-1]
    late = time / end_time >= 0.8
    penicillin = batch['페니실린농도(g/L)'].to_numpy()
    substrate = batch['기질농도(g/L)'].to_numpy()
    our = batch['산소소모율(g/min)'].to_numpy()
    batch_rows.append({
        '배치번호': batch_number,
        '그룹': '정상' if batch_number <= 90 else 'Fault',
        '최종농도': penicillin[-1],
        '농도유지율': penicillin[-1] / penicillin.max(),
        '후기농도기울기': slope(time[late], penicillin[late]),
        'OUR음수비율': np.mean(our < 0),
        'DO평균': batch['용존산소(mg/L)'].mean(),
        'DO최솟값': batch['용존산소(mg/L)'].min(),
        'CO2평균': batch['배가스이산화탄소(%)'].mean(),
        'CO2최댓값': batch['배가스이산화탄소(%)'].max(),
        '기질후기기울기': slope(time[late], substrate[late]),
        'pH표준편차': batch['pH'].std(ddof=1),
        '산총투입량': np.trapezoid(batch['산투입유량(L/h)'], time),
        '염기총투입량': np.trapezoid(batch['염기투입유량(L/h)'], time),
        '총수확량': batch['총수확량(kg)'].iloc[0],
    })

batch_metrics = pd.DataFrame(batch_rows)
display(batch_metrics.groupby('그룹').size().rename('배치수').to_frame())

,배치수
그룹,
Fault,10
정상,90


### 확인

정상 90개와 Fault 10개로 요약됐다. Fault 표본이 작고 두 그룹 크기가 다르므로 Welch t-test와 비모수 Mann–Whitney U를 함께 사용한다.

In [3]:
metrics = [
    '최종농도', '농도유지율', '후기농도기울기', 'OUR음수비율', 'DO평균', 'DO최솟값',
    'CO2평균', 'CO2최댓값', '기질후기기울기', 'pH표준편차', '산총투입량',
    '염기총투입량', '총수확량',
]
rng = np.random.default_rng(42)
rows = []
for metric in metrics:
    normal = batch_metrics.loc[batch_metrics['그룹'].eq('정상'), metric].dropna().to_numpy()
    fault = batch_metrics.loc[batch_metrics['그룹'].eq('Fault'), metric].dropna().to_numpy()
    welch = stats.ttest_ind(fault, normal, equal_var=False)
    mann = stats.mannwhitneyu(fault, normal, alternative='two-sided')
    bootstrap_difference = np.array([
        rng.choice(fault, len(fault), replace=True).mean()
        - rng.choice(normal, len(normal), replace=True).mean()
        for _ in range(3000)
    ])
    rows.append({
        '지표': metric, '정상평균': normal.mean(), 'Fault평균': fault.mean(),
        'Fault-정상': fault.mean() - normal.mean(),
        '95%CI_하한': np.quantile(bootstrap_difference, 0.025),
        '95%CI_상한': np.quantile(bootstrap_difference, 0.975),
        'Welch_p': welch.pvalue, 'Mann_p': mann.pvalue,
        'Hedges_g': hedges_g(fault, normal),
    })

fault_results = pd.DataFrame(rows)
fault_results['Welch_FDR'] = fdr_bh(fault_results['Welch_p'])
fault_results['Mann_FDR'] = fdr_bh(fault_results['Mann_p'])
fault_results['판정'] = np.select(
    [(fault_results['Welch_FDR'] < 0.05) & (fault_results['Mann_FDR'] < 0.05),
     (fault_results['Welch_FDR'] < 0.05) | (fault_results['Mann_FDR'] < 0.05)],
    ['두 검정 유의', '한 검정만 유의'], default='근거 부족',
)
display(fault_results.sort_values('Welch_FDR').round(6))

,지표,정상평균,Fault평균,Fault-정상,95%CI_하한,95%CI_상한,Welch_p,Mann_p,Hedges_g,Welch_FDR,Mann_FDR,판정
1,농도유지율,8.803230e-01,6.384000e-01,-0.241923,-3.690710e-01,-0.097883,0.008691,0.001447,-1.351947,0.037662,0.006269,두 검정 유의
4,DO평균,1.249262e+01,1.329117e+01,0.798551,3.242090e-01,1.275977,0.008110,0.009257,0.947365,0.037662,0.024068,두 검정 유의
10,산총투입량,1.345686e+01,4.555473e+01,32.097874,1.531600e+01,47.524082,0.004183,0.000411,1.843804,0.037662,0.002670,두 검정 유의
0,최종농도,2.507673e+01,1.442134e+01,-10.655387,-1.697351e+01,-3.712722,0.013553,0.005722,-1.316156,0.041558,0.018598,두 검정 유의
8,기질후기기울기,2.160660e-01,4.623930e-01,0.246327,6.584300e-02,0.396664,0.015984,0.044968,0.824815,0.041558,0.064954,한 검정만 유의
11,염기총투입량,1.431046e+04,1.091435e+04,-3396.105650,-5.860434e+03,-690.785861,0.031614,0.013719,-0.847787,0.068497,0.029724,한 검정만 유의
2,후기농도기울기,1.213000e-02,-4.339200e-02,-0.055522,-1.045410e-01,-0.004519,0.061982,0.101572,-0.618319,0.115110,0.132043,근거 부족
3,OUR음수비율,8.615000e-03,1.413000e-02,0.005515,6.500000e-04,0.012582,0.131162,0.024251,1.189483,0.178439,0.039408,한 검정만 유의
6,CO2평균,1.455605e+00,1.352434e+00,-0.103171,-2.280870e-01,0.008144,0.137260,0.019379,-1.035967,0.178439,0.035990,한 검정만 유의
7,CO2최댓값,2.065543e+00,3.252560e+00,1.187017,-1.137120e-01,2.569737,0.122074,0.641690,1.749784,0.178439,0.641690,근거 부족


### 판단

- 두 검정과 FDR 보정에서 모두 차이가 남은 지표는 **농도유지율, DO 평균, 산 총투입량, 최종농도**다.
- Fault는 정상보다 최종농도가 평균 10.66g/L 낮고, 농도유지율이 0.242 낮다. 두 효과크기는 각각 |g|=1.32, 1.35로 크다.
- Fault는 DO 평균이 약 0.80mg/L 높고 산 총투입량이 약 32.10 높다. 이는 이상 운전의 구분 후보지만 인과관계는 아니다.
- OUR 음수비율, pH 변동성, CO₂ 평균 등은 Mann–Whitney만 유의하여 방향성 신호로만 본다.
- 총수확량과 DO 최솟값은 유의한 차이 근거가 없다. Fault 라벨이 반드시 저수확과 일치하지 않는다는 프로젝트 문제와 부합한다.
- Fault가 10개뿐이므로 최종 판단은 효과크기와 신뢰구간, 개별 배치 궤적을 함께 확인해야 한다.